In [10]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash",google_api_key=os.getenv("GOOGLE_API_KEY"))

In [ ]:
from dataclasses import dataclass
import os
import json
from dotenv import load_dotenv

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver

load_dotenv()

# Define system prompt
SYSTEM_PROMPT = """
You are an intelligent AI agent responsible for **analyzing and shortlisting resumes** based on a given job description.

---

### 🎯 Task Overview:
Your primary task is to:
1. Execute the tool **'extract_candidate_details'** to retrieve candidate profiles
2. Analyze each candidate against the job description
3. Score and rank candidates
4. Return ONLY the top 50 candidates in a specific format

---

### ⚙️ Processing Instructions:
1. First, call the 'extract_candidate_details' tool to get all candidate data
2. Analyze each candidate's profile against the **provided job description**
3. Evaluate alignment based on:
   - **Key Skills:** 50%  
   - **Total Experience:** 20%  
   - **Relevant Experience:** 20%  
   - **Notice Period:** 10%
4. Compute a **similarity score (0–100)** for each candidate
5. Rank ALL candidates from highest to lowest score
6. Select and return **only the Top 50 candidates**

---

### 🧾 CRITICAL OUTPUT FORMAT:
Your response must ONLY contain the formatted candidate list. Follow this EXACT format:

1.  
Name: [Full Name]  
Email: [Email Address]  
Contact: [Phone Number]  
CTC: [Current CTC as number]  
ECTC: [Expected CTC as number]  
Resume: [Resume URL]  
Experience: [Years]  
Relevant Experience: [Years]  
Skills: [Comma-separated key skills - max 5-6 most relevant]  
Score: [Numeric score between 0-100]  

2.  
Name: [Full Name]  
Email: [Email Address]  
...

---

### 🧠 CRITICAL GUIDELINES:
- DO NOT include any JSON, code blocks, or markdown tables
- DO NOT include explanations, summaries, or additional text
- DO NOT show the raw tool output
- ONLY return the formatted list of exactly 50 candidates
- Each candidate entry must be numbered (1, 2, 3...)
- Each candidate must have all fields listed above
- Skills should be a SHORT comma-separated list (not the full list)
- Score must be a decimal number (e.g., 94.5, 91.2)
- Sort by Score from highest to lowest

### Example of CORRECT output format:

1.  
Name: Ashish Kumar Rai  
Email: ashishrai190186@gmail.com  
Contact: 9916864964  
CTC: 3500000  
ECTC: 3500000  
Resume: https://example.com/resume.pdf  
Experience: 12  
Relevant Experience: 8  
Skills: Java, HTML, Sitecore CMS, GitHub  
Score: 94.5  

2.  
Name: Priya Sharma  
Email: priyasharma@gmail.com  
Contact: 9876543210  
CTC: 1800000  
ECTC: 2400000  
Resume: https://example.com/resume2.pdf  
Experience: 7  
Relevant Experience: 5  
Skills: Python, Django, REST APIs, Docker  
Score: 91.2  

Remember: Your entire response should be JUST the numbered list - nothing else!
"""

# Define context schema
@dataclass
class Context:
    """Custom runtime context schema."""
    user_id: str

# Predefined path
DEFAULT_RESUME_PATH = "E://resumeagent//resume.json"

@tool
def extract_candidate_details(file_path: str = DEFAULT_RESUME_PATH):
    """
    Reads a resume JSON file and extracts key candidate details.

    Args:
        file_path (str): Path to the JSON file (defaults to predefined path).

    Returns:
        list[dict]: A list of candidate information dictionaries.
    """
    with open(file_path, "r", encoding="utf-8") as file:
        data = json.load(file)

    candidates = data.get("data", {}).get("candidates", [])
    extracted_data = []

    for candidate in candidates:
        candidate_info = {
            "name": candidate.get("name", "N/A"),
            "contact": candidate.get("contactNumber", "N/A"),
            "email": candidate.get("email", "N/A"),
            "resume": candidate.get("resume", "N/A"),
            "key_skills": candidate.get("keySkills", "N/A"),
            "experience": candidate.get("experience", "N/A"),
            "relevant_experience": candidate.get("relevantExperience", "N/A"),
            "current_ctc": candidate.get("currentCtc", "N/A"),
            "expected_ctc": candidate.get("expectedCtc", "N/A"),
            "notice_period": candidate.get("noticePeriod", "N/A"),
        }
        extracted_data.append(candidate_info)
    return extracted_data

# Set up memory
checkpointer = InMemorySaver()

# Create agent
agent = create_agent(
    model=model,
    system_prompt=SYSTEM_PROMPT,
    tools=[extract_candidate_details],
    checkpointer=checkpointer
)

# Run agent
# `thread_id` is a unique identifier for a given conversation.
config = {"configurable": {"thread_id": "0"}}

# Enhanced user message with clearer instructions
user_message = """
Please analyze resumes for the following job requirements:

Responsibilities:
- Design, develop, and deploy AI agents powered by LLMs (e.g., GPT-4, Claude, Gemini)
- Integrate AI agents with tools, APIs, databases, and automation frameworks
- Develop reusable prompt chains and workflows for common tasks and decision-making
- Utilize frameworks like LangChain, AutoGen, CrewAI, or Semantic Kernel to manage multi-agent architectures
- Fine-tune or instruct LLMs for specific use cases and industry applications
- Optimize the performance, reliability, and cost-efficiency of AI workflows
- Collaborate with data scientists, product managers, and engineers to design end-to-end AI solutions
- Implement automation in internal tools, customer interactions, or operational pipelines using AI agents

Requirements:
- Extensive experience with LLMs such as OpenAI GPT, Anthropic Claude, or Meta Llama
- Hands-on experience with agentic frameworks (LangChain, AutoGen, CrewAI, etc.)
- Proficiency in Python and relevant AI libraries (e.g., HuggingFace, Transformers, LangChain)
- Strong understanding of prompt engineering and retrieval-augmented generation (RAG)
- Knowledge of automation tools like Zapier, Make, Airflow, or custom Python automation
- Experience working with APIs, webhooks, and data integrations

Nice-to-Have:
- Experience with vector databases (e.g., Pinecone, Weaviate, FAISS)
- Knowledge of fine-tuning or customizing open-source LLMs
- Familiarity with cloud platforms (AWS, GCP, Azure) and deployment of AI solutions
- Experience with UI/UX for chatbot or agent interfaces

Desired Skills: .NET, Python, ASP.NET, C#, Flask

Please provide the top 50 candidates in the specified format.
"""

response = agent.invoke(
    {"messages": [{"role": "user", "content": user_message}]},
    config=config,
    context=Context(user_id="0")
)

# The correct way to access the response
print("=" * 80)
print("AGENT RESPONSE:")
print("=" * 80)

# Extract the final message
if "messages" in response:
    last_message = response["messages"][-1]

    # Handle both AIMessage object or dict
    if hasattr(last_message, "content"):
        content = last_message.content
    elif isinstance(last_message, dict) and "content" in last_message:
        content = last_message["content"]
    else:
        content = str(last_message)

    # Handle case where content is a list of dicts like [{'type': 'text', 'text': '...'}]
    if isinstance(content, list):
        formatted_text = ""
        for item in content:
            if isinstance(item, dict) and "text" in item:
                formatted_text += item["text"]
            else:
                formatted_text += str(item)
        content = formatted_text

    # Replace escaped newlines and clean up formatting
    formatted_output = (
        content.replace("\\n", "\n")
        .replace("\\t", "\t")
        .replace("\\", "")
        .strip()
    )

    print(formatted_output)

else:
    print("Full response structure:")
    print(response)


AGENT RESPONSE:
1.
Name: VISHAL MANOCHA
Email: vishal.manocha2010@gmail.com
Contact: 7060398298
CTC: 2200000
ECTC: 2200000
Resume: https://irecruit.intelliswift.com/system/resumes/resume_docs/000/000/047/original/Naukri_VISHALMANOCHA_8y_0m_.pdf
Experience: 10
Relevant Experience: 5
Skills: Large Language Model, Artificial Intelligence, Machine Learning, Natural Language Processing, Python, Azure
Score: 95.0

2.
Name: KIRANJEET KAUR
Email: khairakiran6@gmail.com
Contact: 9910335459
CTC: 2200000
ECTC: 2200000
Resume: https://irecruit.intelliswift.com/system/resumes/resume_docs/000/000/031/original/Naukri_kiranjeetkaur_11y_6m_.pdf
Experience: 10
Relevant Experience: 5
Skills: Generative Artificial Intelligence, Natural Language Processing, Machine Learning, Python, Azure, Aws
Score: 95.0

3.
Name: MANDEEP KAUR
Email: mandeepkaur.mca07@gmail.com
Contact: 8283818805
CTC: 2200000
ECTC: 2200000
Resume: https://irecruit.intelliswift.com/system/resumes/resume_docs/000/000/030/original/Naukri_Ma

In [13]:
import os
import json
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.tools import tool
import os
from typing import Literal
import json
from pydantic import BaseModel, Field
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()

# Define system prompt
SYSTEM_PROMPT = """
You are an intelligent AI agent responsible for **analyzing and shortlisting resumes** based on a given job description.

---

### 🎯 Task Overview:
Your primary task is to:
1. Execute the tool **'extract_candidate_details'** to retrieve candidate profiles
2. Analyze each candidate against the job description
3. Score and rank candidates
4. Return ONLY the top 50 candidates in a specific format

---

### ⚙️ Processing Instructions:
1. First, call the 'extract_candidate_details' tool to get all candidate data
2. Analyze each candidate's profile against the **provided job description**
3. Evaluate alignment based on:
   - **Key Skills:** 50%  
   - **Total Experience:** 20%  
   - **Relevant Experience:** 20%  
   - **Notice Period:** 10%
4. Compute a **similarity score (0–100)** for each candidate
5. Rank ALL candidates from highest to lowest score
6. Select and return **only the Top 50 candidates**

---

### 🧾 CRITICAL OUTPUT FORMAT:
Your response must ONLY contain the formatted candidate list. Follow this EXACT format:

1.  
Name: [Full Name]  
Email: [Email Address]  
Contact: [Phone Number]  
CTC: [Current CTC as number]  
ECTC: [Expected CTC as number]  
Resume: [Resume URL]  
Experience: [Years]  
Relevant Experience: [Years]  
Skills: [Comma-separated key skills - max 5-6 most relevant]  
Score: [Numeric score between 0-100]  

2.  
Name: [Full Name]  
Email: [Email Address]  
...

---

### 🧠 CRITICAL GUIDELINES:
- DO NOT include any JSON, code blocks, or markdown tables
- DO NOT include explanations, summaries, or additional text
- DO NOT show the raw tool output
- ONLY return the formatted list of exactly 50 candidates
- Each candidate entry must be numbered (1, 2, 3...)
- Each candidate must have all fields listed above
- Skills should be a SHORT comma-separated list (not the full list)
- Score must be a decimal number (e.g., 94.5, 91.2)
- Sort by Score from highest to lowest

### Example of CORRECT output format:

1.  
Name: Ashish Kumar Rai  
Email: ashishrai190186@gmail.com  
Contact: 9916864964  
CTC: 3500000  
ECTC: 3500000  
Resume: https://example.com/resume.pdf  
Experience: 12  
Relevant Experience: 8  
Skills: Java, HTML, Sitecore CMS, GitHub  
Score: 94.5  

2.  
Name: Priya Sharma  
Email: priyasharma@gmail.com  
Contact: 9876543210  
CTC: 1800000  
ECTC: 2400000  
Resume: https://example.com/resume2.pdf  
Experience: 7  
Relevant Experience: 5  
Skills: Python, Django, REST APIs, Docker  
Score: 91.2  

Remember: Your entire response should be JUST the numbered list - nothing else!
"""

# Predefined path
DEFAULT_RESUME_PATH = "E://resumeagent//resume.json"

@tool
def extract_candidate_details(file_path: str = DEFAULT_RESUME_PATH):
    """
    Reads a resume JSON file and extracts key candidate details.

    Args:
        file_path (str): Path to the JSON file (defaults to predefined path).

    Returns:
        list[dict]: A list of candidate information dictionaries.
    """
    with open(file_path, "r", encoding="utf-8") as file:
        data = json.load(file)

    candidates = data.get("data", {}).get("candidates", [])
    extracted_data = []

    for candidate in candidates:
        candidate_info = {
            "name": candidate.get("name", "N/A"),
            "contact": candidate.get("contactNumber", "N/A"),
            "email": candidate.get("email", "N/A"),
            "resume": candidate.get("resume", "N/A"),
            "key_skills": candidate.get("keySkills", "N/A"),
            "experience": candidate.get("experience", "N/A"),
            "relevant_experience": candidate.get("relevantExperience", "N/A"),
            "current_ctc": candidate.get("currentCtc", "N/A"),
            "expected_ctc": candidate.get("expectedCtc", "N/A"),
            "notice_period": candidate.get("noticePeriod", "N/A"),
        }
        extracted_data.append(candidate_info)
    return extracted_data

### The below is model ##

model = ChatGoogleGenerativeAI(model="gemini-2.5-flash",google_api_key=os.getenv("GOOGLE_API_KEY"))

# Create agent
agent = create_agent(
    model=model,
    system_prompt=SYSTEM_PROMPT,
    tools=[extract_candidate_details],
)

# Enhanced user message with clearer instructions
user_message = "Join our Global Medical Affairs team as a Senior Medical Data Scientist, where you will design and implement data science solutions using advanced statistical methods, predictive modeling, and machine learning. Your insights will drive scientific engagement, strategic decision-making, and patient-focused strategies. As a key member of the Medical Data & Analytics (MDnA) team, you will transform complex data into actionable intelligence to inform medical strategy and improve patient outcomes. This hands-on role requires analytical depth and strong problem-solving skills. As an ideal candidate, you are an advanced problem solver who combines deep data science expertise, hands-on technical skills, and the ability to mentor and motivate peers. Expand your expertise and innovate with cutting-edge analytics like large language models (LLMs) and Generative AI. Key Responsibilities: * Design and build high-impact projects addressing complex medical questions across therapeutic areas, applying advanced data science methods. * Conduct end-to-end data science projects across structured and unstructured datasets using Python, R, SQL, Tableau, Power BI, or similar platforms. * Design, apply, and evaluate test-and-learn approaches, ensuring methodological rigor and actionable insights. * Partner with business leaders to define project scope and conduct medical data science projects, translating complex data into insights for more effective field medical activities. * Design and optimize scalable data pipelines in collaboration with engineering teams, ensuring reproducibility, robustness, and compliance. * Translate complex model outputs into clear insights and deliver compelling presentations to leadership teams. * Serve as a technical subject matter expert in data science, influencing design and methodology choices across the MDnA team. * Continuously learn, evaluate, and introduce emerging technologies, strengthening Amgen’s data science toolbox. * Maintain best practices in reproducibility, documentation, version control, and model governance. Basic Qualifications: * PhD in Computer Science, Engineering, Statistics, or a related field with 2+ years of data analytics or data science experience; OR, * Master’s degree in Computer Science, Engineering, Statistics, or a related field with 5+ years of data analytics or data science experience; OR, * Bachelor’s degree in Computer Science, Engineering, Statistics, or a related field with 7+ years of data analytics or data science experience. * Experience in data science, statistics, machine learning, or advanced analytics, with hands-on application of statistical techniques. * Demonstrated expertise in data science, statistical modeling, machine learning, advanced predictive modeling, NLP, or related fields. * Advanced programming in SQL and Python (R a plus); experience in Databricks or similar environments. * Strong storytelling and presentation skills. Preferred Qualifications: * Life sciences or pharma experience with data science applications. * Deep understanding of machine learning frameworks (e.g., scikit-learn, TensorFlow, PyTorch). * Practical experience with LLMs, RAG architectures, or advanced NLP applications. * Familiarity with MLflow, AWS, Git, and DevOps-style model deployment workflows. * Track record of peer mentorship and leadership within technical teams. Soft Skills: * Curiosity, adaptability, and a passion for continuous learning. * Motivational influence and ability to inspire peers. * Strong problem-solving and critical thinking. * Effective communication skills. * Collaborative mindset."


response = agent.invoke(
    {"messages": [{"role": "user", "content": user_message}]},
)

# The correct way to access the response
print("=" * 80)
print("AGENT RESPONSE:")
print("=" * 80)

# Handle possible response types
if isinstance(response, dict):
    # Common case: response contains 'output' or 'final_output'
    content = (
        response.get("output")
        or response.get("final_output")
        or response.get("messages", "")
        or str(response)
    )
elif hasattr(response, "content"):
    # AIMessage or similar object
    content = response.content
else:
    # String or unknown type
    content = str(response)

# Handle nested content structures
if isinstance(content, list):
    formatted_text = ""
    for item in content:
        if isinstance(item, dict) and "text" in item:
            formatted_text += item["text"]
        else:
            formatted_text += str(item)
    content = formatted_text

# Clean up escaped characters
formatted_output = (
    content.replace("\\n", "\n")
    .replace("\\t", "\t")
    .replace("\\", "")
    .strip()
)

print(formatted_output)


AGENT RESPONSE:
content='Join our Global Medical Affairs team as a Senior Medical Data Scientist, where you will design and implement data science solutions using advanced statistical methods, predictive modeling, and machine learning. Your insights will drive scientific engagement, strategic decision-making, and patient-focused strategies. As a key member of the Medical Data & Analytics (MDnA) team, you will transform complex data into actionable intelligence to inform medical strategy and improve patient outcomes. This hands-on role requires analytical depth and strong problem-solving skills. As an ideal candidate, you are an advanced problem solver who combines deep data science expertise, hands-on technical skills, and the ability to mentor and motivate peers. Expand your expertise and innovate with cutting-edge analytics like large language models (LLMs) and Generative AI. Key Responsibilities: * Design and build high-impact projects addressing complex medical questions across the

In [41]:
prompts = """
You are an intelligent AI agent responsible for **analyzing and shortlisting resumes** based on a given job description.

---

### 🎯 Task Overview:
Your primary task is to:
1. Execute the tool **'extract_candidate_details'** to retrieve candidate profiles
2. Analyze each candidate against the job description
3. Score and rank candidates
4. Return ONLY the top 50 candidates in a specific format

---

### ⚙️ Processing Instructions:
1. First, call the 'extract_candidate_details' tool to get all candidate data
2. Analyze each candidate's profile against the **provided job description**
3. Evaluate alignment based on:
   - **Key Skills:** 50%  
   - **Total Experience:** 20%  
   - **Relevant Experience:** 20%  
   - **Notice Period:** 10%
4. Compute a **similarity score (0–100)** for each candidate
5. Rank ALL candidates from highest to lowest score
6. Select and return **only the Top 50 candidates**

---

### 🧾 CRITICAL OUTPUT FORMAT:
Your response must ONLY contain the formatted candidate list. Follow this EXACT format:

1.  
Name: [Full Name]  
Email: [Email Address]  
Contact: [Phone Number]  
CTC: [Current CTC as number]  
ECTC: [Expected CTC as number]  
Resume: [Resume URL]  
Experience: [Years]  
Relevant Experience: [Years]  
Skills: [Comma-separated key skills - max 5-6 most relevant]  
Score: [Numeric score between 0-100]  

2.  
Name: [Full Name]  
Email: [Email Address]  
...

---

### 🧠 CRITICAL GUIDELINES:
- DO NOT include any JSON, code blocks, or markdown tables
- DO NOT include explanations, summaries, or additional text
- DO NOT show the raw tool output
- ONLY return the formatted list of exactly 50 candidates
- Each candidate entry must be numbered (1, 2, 3...)
- Each candidate must have all fields listed above
- Skills should be a SHORT comma-separated list (not the full list)
- Score must be a decimal number (e.g., 94.5, 91.2)
- Sort by Score from highest to lowest

### Example of CORRECT output format:

1.  
Name: Ashish Kumar Rai  
Email: ashishrai190186@gmail.com  
Contact: 9916864964  
CTC: 3500000  
ECTC: 3500000  
Resume: https://example.com/resume.pdf  
Experience: 12  
Relevant Experience: 8  
Skills: Java, HTML, Sitecore CMS, GitHub  
Score: 94.5  

2.  
Name: Priya Sharma  
Email: priyasharma@gmail.com  
Contact: 9876543210  
CTC: 1800000  
ECTC: 2400000  
Resume: https://example.com/resume2.pdf  
Experience: 7  
Relevant Experience: 5  
Skills: Python, Django, REST APIs, Docker  
Score: 91.2  

Remember: Your entire response should be JUST the numbered list - nothing else!
"""

In [22]:
from langchain.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
import os
# Define a very simple tool function that returns the current time
# Predefined path
# DEFAULT_RESUME_PATH = "E://resumeagent//resume.json"
DEFAULT_RESUME_PATH = "E://resumeagent//candidates_filtered.json"

@tool
def extract_candidate_details(file_path: str = DEFAULT_RESUME_PATH):
    """
    Reads a resume JSON file and extracts key candidate details.

    Args:
        file_path (str): Path to the JSON file (defaults to predefined path).

    Returns:
        list[dict]: A list of candidate information dictionaries.
    """
    with open(file_path, "r", encoding="utf-8") as file:
        data = json.load(file)

    candidates = data.get("data", {}).get("candidates", [])
    extracted_data = []

    for candidate in candidates:
        candidate_info = {
            "name": candidate.get("name", "N/A"),
            "contact": candidate.get("contactNumber", "N/A"),
            "email": candidate.get("email", "N/A"),
            "resume": candidate.get("resume", "N/A"),
            "key_skills": candidate.get("keySkills", "N/A"),
            "experience": candidate.get("experience", "N/A"),
            "relevant_experience": candidate.get("relevantExperience", "N/A"),
            "current_ctc": candidate.get("currentCtc", "N/A"),
            "expected_ctc": candidate.get("expectedCtc", "N/A"),
            "notice_period": candidate.get("noticePeriod", "N/A"),
        }
        extracted_data.append(candidate_info)
    return extracted_data

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(model="gemini-2.5-flash",google_api_key=os.getenv("GEMINI_API_KEY"),timeout=3000,streaming=True)

In [30]:
from dataclasses import dataclass

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.tools import tool, ToolRuntime
from langgraph.checkpoint.memory import InMemorySaver


# Define system prompt
SYSTEM_PROMPT = """ You are an intelligent AI agent responsible for **analyzing and shortlisting resumes** based on a given job description.

        ---

        ### 🎯 Task Overview:
        Your primary task is to:
        1. Execute the tool **'extract_candidate_details'** to retrieve candidate profiles
        2. Analyze each candidate against the job description
        3. Score and rank candidates
        4. Return ONLY the top 50 candidates in a specific format

        ---

        ### ⚙️ Processing Instructions:
        1. First, call the 'extract_candidate_details' tool to get all candidate data
        2. Analyze each candidate's profile against the **provided job description**
        3. Evaluate alignment based on:
        - **Key Skills:** 50%  
        - **Total Experience:** 20%  
        - **Relevant Experience:** 20%  
        - **Notice Period:** 10%
        4. Compute a **similarity score (0–100)** for each candidate
        5. Rank ALL candidates from highest to lowest score
        6. Select and return **only the Top 50 candidates**

        ---

        ### 🧾 CRITICAL OUTPUT FORMAT:
        Your response must ONLY contain the formatted candidate list. Follow this EXACT format:

        1.  
        Name: [Full Name]  
        Email: [Email Address]  
        Contact: [Phone Number]  
        CTC: [Current CTC as number]  
        ECTC: [Expected CTC as number]  
        Resume: [Resume URL]  
        Experience: [Years]  
        Relevant Experience: [Years]  
        Skills: [Comma-separated key skills - max 5-6 most relevant]  
        Score: [Numeric score between 0-100]  

        2.  
        Name: [Full Name]  
        Email: [Email Address]  
        ...

        ---

        ### 🧠 CRITICAL GUIDELINES:
        - DO NOT include any JSON, code blocks, or markdown tables
        - DO NOT include explanations, summaries, or additional text
        - DO NOT show the raw tool output
        - ONLY return the formatted list of exactly 50 candidates
        - Each candidate entry must be numbered (1, 2, 3...)
        - Each candidate must have all fields listed above
        - Skills should be a SHORT comma-separated list (not the full list)
        - Score must be a decimal number (e.g., 94.5, 91.2)
        - Sort by Score from highest to lowest

        ### Example of CORRECT output format:

        1.  
        Name: Ashish Kumar Rai  
        Email: ashishrai190186@gmail.com  
        Contact: 9916864964  
        CTC: 3500000  
        ECTC: 3500000  
        Resume: https://example.com/resume.pdf  
        Experience: 12  
        Relevant Experience: 8  
        Skills: Java, HTML, Sitecore CMS, GitHub  
        Score: 94.5  

        2.  
        Name: Priya Sharma  
        Email: priyasharma@gmail.com  
        Contact: 9876543210  
        CTC: 1800000  
        ECTC: 2400000  
        Resume: https://example.com/resume2.pdf  
        Experience: 7  
        Relevant Experience: 5  
        Skills: Python, Django, REST APIs, Docker  
        Score: 91.2  

        Remember: Your entire response should be JUST the numbered list - nothing else!
        """,

# Define context schema
@dataclass
class Context:
    """Custom runtime context schema."""
    user_id: str

# Define tools
DEFAULT_RESUME_PATH = "E://resumeagent//candidates_filtered.json"

@tool
def extract_candidate_details(file_path: str = DEFAULT_RESUME_PATH):
    """
    Reads a resume JSON file and extracts key candidate details.

    Args:
        file_path (str): Path to the JSON file (defaults to predefined path).

    Returns:
        list[dict]: A list of candidate information dictionaries.
    """
    with open(file_path, "r", encoding="utf-8") as file:
        data = json.load(file)

    candidates = data.get("data", {}).get("candidates", [])
    extracted_data = []

    for candidate in candidates:
        candidate_info = {
            "name": candidate.get("name", "N/A"),
            "contact": candidate.get("contactNumber", "N/A"),
            "email": candidate.get("email", "N/A"),
            "resume": candidate.get("resume", "N/A"),
            "key_skills": candidate.get("keySkills", "N/A"),
            "experience": candidate.get("experience", "N/A"),
            "relevant_experience": candidate.get("relevantExperience", "N/A"),
            "current_ctc": candidate.get("currentCtc", "N/A"),
            "expected_ctc": candidate.get("expectedCtc", "N/A"),
            "notice_period": candidate.get("noticePeriod", "N/A"),
        }
        extracted_data.append(candidate_info)
    return extracted_data

# Configure model
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash",google_api_key=os.getenv("GEMINI_API_KEY"),timeout=3000,streaming=True)

# Define response format
@dataclass
class CandidateInfo(BaseModel):
    name: str = Field(..., description="Full name of the candidate")
    email: str = Field(..., description="Email address of the candidate")
    contact: str = Field(..., description="Contact phone number of the candidate")
    ctc: float = Field(..., description="Current CTC in numeric form")
    ectc: float = Field(..., description="Expected CTC in numeric form")
    resume: str = Field(..., description="Resume URL of the candidate")
    experience: float = Field(..., description="Total years of experience")
    relevant_experience: float = Field(..., description="Years of relevant experience")
    skills: str = Field(..., description="Comma-separated list of top 5-6 key skills")
    score: float = Field(..., description="Agent generated similarity score between 0 and 100")

# For structured output, return a list
class CandidateList(BaseModel):
    candidates: list[CandidateInfo] = Field(..., description="List of top 50 candidates")

# Set up memory
checkpointer = InMemorySaver()

# Create agent
agent = create_agent(
    model=model,
    system_prompt=SYSTEM_PROMPT,
    tools=[extract_candidate_details],
    context_schema=Context,
    response_format=CandidateList,
    checkpointer=checkpointer
)

# Run agent
# `thread_id` is a unique identifier for a given conversation.
config = {"configurable": {"thread_id": "1"}}


response = agent.invoke({
    "messages": [{"role": "user", "content": "Join our Global Medical Affairs team as a Senior Medical Data Scientist, where you will design and implement data science solutions using advanced statistical methods, predictive modeling, and machine learning. Your insights will drive scientific engagement, strategic decision-making, and patient-focused strategies.\n\nAs a key member of the Medical Data & Analytics (MDnA) team, you will transform complex data into actionable intelligence to inform medical strategy and improve patient outcomes. This hands-on role requires analytical depth and strong problem-solving skills. As an ideal candidate, you are an advanced problem solver who combines deep data science expertise, hands-on technical skills, and the ability to mentor and motivate peers. Expand your expertise and innovate with cutting-edge analytics like large language models (LLMs) and Generative AI.\n\nKey Responsibilities:\n\n* Design and build high-impact projects addressing complex medical questions across therapeutic areas, applying advanced data science methods.\n* Conduct end-to-end data science projects across structured and unstructured datasets using Python, R, SQL, Tableau, Power BI, or similar platforms.\n* Design, apply, and evaluate test-and-learn approaches, ensuring methodological rigor and actionable insights.\n* Partner with business leaders to define project scope and conduct medical data science projects, translating complex data into insights for more effective field medical activities.\n* Design and optimize scalable data pipelines in collaboration with engineering teams, ensuring reproducibility, robustness, and compliance.\n* Translate complex model outputs into clear insights and deliver compelling presentations to leadership teams.\n* Serve as a technical subject matter expert in data science, influencing design and methodology choices across the MDnA team.\n* Continuously learn, evaluate, and introduce emerging technologies, strengthening Amgen’s data science toolbox.\n* Maintain best practices in reproducibility, documentation, version control, and model governance.\n\nBasic Qualifications:\n\n* PhD in Computer Science, Engineering, Statistics, or a related field with 2+ years of data analytics or data science experience; OR,\n* Master’s degree in Computer Science, Engineering, Statistics, or a related field with 5+ years of data analytics or data science experience; OR,\n* Bachelor’s degree in Computer Science, Engineering, Statistics, or a related field with 7+ years of data analytics or data science experience.\n* Experience in data science, statistics, machine learning, or advanced analytics, with hands-on application of statistical techniques.\n* Demonstrated expertise in data science, statistical modeling, machine learning, advanced predictive modeling, NLP, or related fields.\n* Advanced programming in SQL and Python (R a plus); experience in Databricks or similar environments.\n* Strong storytelling and presentation skills.\n\nPreferred Qualifications:\n\n* Life sciences or pharma experience with data science applications.\n* Deep understanding of machine learning frameworks (e.g., scikit-learn, TensorFlow, PyTorch).\n* Practical experience with LLMs, RAG architectures, or advanced NLP applications.\n* Familiarity with MLflow, AWS, Git, and DevOps-style model deployment workflows.\n* Track record of peer mentorship and leadership within technical teams.\n\nSoft Skills:\n\n* Curiosity, adaptability, and a passion for continuous learning.\n* Motivational influence and ability to inspire peers.\n* Strong problem-solving and critical thinking.\n* Effective communication skills.\n* Collaborative mindset"}]},
    config=config,
    context=Context(user_id="1")
)

print(response['structured_response'])
# ResponseFormat(
#     punny_response="You're 'thund-erfully' welcome! It's always a 'breeze' to help you stay 'current' with the weather. I'm just 'cloud'-ing around waiting to 'shower' you with more forecasts whenever you need them. Have a 'sun-sational' day in the Florida sunshine!",
#     weather_conditions=None
# )

KeyError: 'structured_response'

In [ ]:
# import json

# # Load the JSON file
# with open('resume.json', 'r') as f:
#     data = json.load(f)

# # 1. Check how many candidates exist
# total_candidates = len(data['data']['candidates'])
# print(f"Total candidates: {total_candidates}")

# # 2. Drop 1500 candidates (keeping only the rest)
# if total_candidates > 1500:
#     # Keep candidates from index 1500 onwards (drops first 1500)
#     data['data']['candidates'] = data['data']['candidates'][1500:]
#     print(f"Remaining candidates: {len(data['data']['candidates'])}")
# else:
#     print(f"Cannot drop 1500 candidates - only {total_candidates} available")

# # Save the modified JSON
# with open('candidates_filtered.json', 'w') as f:
#     json.dump(data, f, indent=4)


Total candidates: 1806
Remaining candidates: 306
